# Rate Limiter simple. 

- Design rate limiter in memory 

- N requests per second

- multiple users but all separate

# 1. Static Entity Category

Define a small category ( \mathcal{D} ) representing the domain schema.

Objects:

$$
\mathrm{Ob}(\mathcal{D})
=

{
User,
Request,
Decision,
LimiterState
}
$$

Morphisms:

$$
owner : Request \to User
$$

$$
decision_for : Decision \to Request
$$

The morphism

$$
owner : Request \to User
$$

encodes the one-to-many relation:

$$
owner^{-1}(u)
=

{r \in Request \mid owner(r)=u}
$$

This fiber is the set of requests belonging to user (u).

Thus a "user owns many requests" is not a product or coproduct.

It is a morphism together with its fibers.

---

# 2. Indexed Limiter State

Each user possesses an individual limiter state:

$$
S_u
$$

Examples:

* token bucket count
* refill timestamp
* sliding window counters

The total state of the system is the product over users:

$$
S
=

\prod_{u \in User}
S_u
$$

or categorically:

$$
S
=

\Pi_{u : User} S_u
$$

This is an indexed product.

This is the first actual universal property appearing in the system.

For every family of morphisms

$$
f_u : X \to S_u
$$

there exists a unique morphism

$$
f : X \to S
$$

such that

$$
\pi_u \circ f = f_u
$$

for every user (u).

---

# 3. Event Space

Requests occur through time.

Define a time object:

$$
T
$$

A request stream is a morphism

$$
R : T \to Request
$$

or equivalently an element of

$$
Request^T
$$

This is an exponential object in a cartesian closed category.

If time is discrete:

$$
R : \mathbb{N} \to Request
$$

then the request stream is simply a sequence.

---

# 4. Rate Limiter as State Transition System

Define:

$$
E := Request
$$

$$
O := Decision
$$

$$
S := \prod_u S_u
$$

The transition function is

$$
\delta :
S \times E
\to
S \times O
$$

Explicitly:

$$
\delta(s,r)
=

(s',o)
$$

where:

* (s) is current limiter state,
* (r) is incoming request,
* (s') is updated state,
* (o) is allow or reject.

This is the standard deterministic automaton form.

---

# 5. Coalgebra Formulation

Let

$$
F(X)
=

(O \times X)^E
$$

Then a rate limiter is an (F)-coalgebra:

$$
c : S \to F(S)
$$

or equivalently:

$$
c :
S
\to
(O \times S)^E
$$

Given state (s), the coalgebra returns a function:

$$
c(s)
:
E
\to
O \times S
$$

meaning:

"given a request, produce a decision and next state."

This is the canonical coalgebraic representation of an interactive system.

---

# 6. User-Indexed Coalgebra

Most implementations only modify the state for the requesting user.

Define:

$$
lookup :
Request \to User
$$

Then:

$$
\delta_u :
S_u \times Request_u
\to
S_u \times Decision
$$

The global transition becomes:

$$
\delta
:
\left(
\prod_u S_u
\right)
\times Request
\to
\left(
\prod_u S_u
\right)
\times Decision
$$

with

$$
\pi_v(s')
=

\pi_v(s)
\qquad
v \neq owner(r)
$$

and

$$
\pi_{owner(r)}(s')
=

\delta_{owner(r)}
(
\pi_{owner(r)}(s),
r
)
$$

Thus each request updates only one coordinate of the product object.

---

# 7. State Monad View

The transition can be curried:

$$
Request
\to
(S \to S \times Decision)
$$

which is exactly:

$$
Request
\to
State(S,Decision)
$$

where

$$
State(S,X)
=

S \to S \times X
$$

Thus rate limiting middleware naturally inhabits the state monad.

This explains why middleware chains compose so naturally.

---

# 8. Sliding Window Example

For a sliding window limiter:

$$
S_u
=

List(Timestamp)
$$

Transition:

$$
\delta_u :
List(Timestamp)
\times
Request
\to
List(Timestamp)
\times
Decision
$$

Algorithmically:

1. remove expired timestamps,
2. count remaining timestamps,
3. decide allow/reject,
4. append current timestamp if allowed.

Categorically this is still the same coalgebra:

$$
S_u \times Request
\to
S_u \times Decision
$$

only the internal state object changes.

---

# 9. Universal Properties Present

The construction contains several universal properties simultaneously.

## Product

$$
S
=

\prod_u S_u
$$

Global limiter state.

---

## Pullback/Fiber

$$
Request_u
=

Request
\times_{User}
{u}
$$

Requests belonging to user (u).

---

## Exponential Object

$$
Request^T
$$

The space of all request streams.

---

## Coalgebra

$$
S
\to
(O \times S)^E
$$

Interactive behaviour of the limiter.

---

# 10. Final Categorical Picture

$$
Request
\xrightarrow{owner}
User
$$

$$
S
=

\prod_u S_u
$$

$$
\delta :
S \times Request
\to
S \times Decision
$$

or equivalently

$$
S
\to
(Decision \times S)^{Request}
$$

The ownership relation is a morphism.

The per-user state storage is a product.

The request history is an exponential object.

The running limiter is a coalgebra.


## Entities:

- User
- UsersService
- RateLimiter
- UserRequests

## Model Relationships 

UsersService one to many of users, RateLimiter one to many of userrequests tracking. We can use simple dependency injection for now

## APIs and interfaces:

- check(user_id: str) -> bool
- call(user_id: str) -> bool, state
- reset(user_id: str) -> None

## State Machine 

Ratelimiter per user.
- Open
- Closed

While requests should have:
- Pending
- Approved
- Rejected

Transitions are all linear for ratelimiter:

Open -> Closed
Closed -> Open

Transitions are all linear for requests:

Pending -> Approved
Pending -> Rejected

## Ownership

- User own their own id
- Users Service owns the collection of users. Equivalent to the collection of objects in the category

- RateLimiter owns the state machine aggregate (the category) and the rate limiting logic/ Mutation

- OpenState should own the transitions of the ratelimiter and resolution logic

- UserRequest own their own id and their own status of the workflow




In [ ]:
from enum import Enum

class UserRequestState:
    PENDING = 1
    APPROVED = 2
    REJECTED = 3

class UserRateLimitState:
    @staticmethod
    def resolve():
        pass

class User:
    def __init__(self, user_id):
        self.user_id = user_id

class UserRequest:
    def __init__(self, user_id):
        self.user_id = user_id
        self.status = UserRequestState.PENDING

class UserRatelimit:
    def __init__(self, user_id, strategy):
        self.user_id = user_id

class RateLimitStrategy:
    def __init__(self):
        self.user_ratelimits = 

class RateLimiter:
    def __init__(self, users):
        self.users = users
        self.strategy = strategy
        for user in users:
            self.strategy.user_ratelimits[user.user_id] = UserRatelimit(user.user_id)
       

    def resolve(self):
        return self.strategy.resolve()